In [0]:
from pyspark.sql import functions as F

# 1. Cargamos nuestra fuente principal de transacciones (SUNAT) y las dimensiones maestras
df_silver_sunat = spark.table("silver.sunat_paises")

dim_tiempo = spark.table("gold.dim_tiempo")
dim_pais = spark.table("gold.dim_pais")
dim_sector = spark.table("gold.dim_producto_sector")

# 2. Hacemos los JOINs para cambiar los textos por los IDs numéricos (Surrogate Keys)
df_hechos = df_silver_sunat \
    .join(dim_pais, on="pais_destino", how="left") \
    .join(dim_sector, on="sector", how="left") \
    .join(dim_tiempo, on="fecha", how="left")

# 3. Seleccionamos solo las claves y las métricas (Regla estricta del modelado dimensional)
# Manejamos los nulos (coalesce a -1) por si en el futuro entra un dato sin dimensión mapeada
df_hechos_final = df_hechos.select(
    F.coalesce(F.col("id_tiempo"), F.lit(-1)).alias("id_tiempo"),
    F.coalesce(F.col("id_pais"), F.lit(-1)).alias("id_pais"),
    F.coalesce(F.col("id_sector"), F.lit(-1)).alias("id_sector"),
    F.round(F.col("valor_fob_usd"), 2).alias("valor_fob_usd"),
    F.round(F.col("valor_fob_pen"), 2).alias("valor_fob_pen")
)

# 4. Guardamos la tabla de hechos
df_hechos_final.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.hechos_exportaciones")

print(f"Tabla de Hechos generada exitosamente con {df_hechos_final.count()} transacciones.")
display(df_hechos_final.limit(10))

Tabla de Hechos generada exitosamente con 2148 transacciones.


id_tiempo,id_pais,id_sector,valor_fob_usd,valor_fob_pen
20050101,37,2,5.936186992E10,1.9512246642704E11
20050101,15,2,7.23038701E9,2.376628210187E10
20050101,48,2,4.0010116E8,1.31513251292E9
20050101,134,2,2.49303056E9,8.19459145072E9
20050101,174,2,3.28697997E9,1.080430316139E10
20050101,143,2,1.40570073E9,4.62053829951E9
20050101,61,2,9.4764116E8,3.11489649292E9
20050101,18,2,1.7921203E8,5.8906994261E8
20050101,74,2,8.2033E7,2.69642471E8
20050101,77,2,0.0,0.0
